# Module 05 — Matrix Multiplication on GPUs

This notebook accompanies [the Module 05 lesson](../../docs/modules/05-matrix-multiplication.md). It measures GEMM on one NVIDIA T4 and connects the results to tiling, Tensor Cores, arithmetic intensity, and LLM inference.

Run it on a Lightning AI GPU runtime. The default experiment uses FP16 square matrices from 256 through 4096 and projection shapes with 1, 4, 16, 64, 256, and 1024 token rows.

## Setup

Inputs stay on the GPU during timing. CUDA events measure device-side elapsed time; a median over repeated samples reduces noise from one unusually slow launch.

In [ ]:
import statistics

import torch

assert torch.cuda.is_available(), 'Select a CUDA runtime, preferably an NVIDIA T4.'
device = torch.device('cuda:0')
props = torch.cuda.get_device_properties(device)
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', props.name)
print('Compute capability:', f'{props.major}.{props.minor}')
print('VRAM:', round(props.total_memory / 2**30, 2), 'GiB')

## 1. GEMM sanity check

For $A\in\mathbb{R}^{M\times K}$ and $B\in\mathbb{R}^{K\times N}$, the output has shape $M\times N$ and the conventional operation count is $2MNK$ FLOPs.

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]], device=device)
b = torch.tensor([[5., 6.], [7., 8.]], device=device)
c = a @ b
print(c.cpu())
assert torch.equal(c.cpu(), torch.tensor([[19., 22.], [43., 50.]]))

## 2. Tile-size model

For a square tile of width $T$, loading FP32 $A$ and $B$ tiles costs $8T^2$ bytes while the tile multiplication performs $2T^3$ FLOPs. The local teaching-model intensity is therefore $T/4$ FLOPs/byte. This excludes output traffic and whole-GEMM cache behavior.

A one-thread-per-output $64\times64$ block would need 4,096 threads, so real kernels use hierarchical tiling and multiple outputs per thread.

In [ ]:
def print_tile_model(tile_sizes=(8, 16, 32, 64)):
    print('Tile | Input storage | FLOPs | Arithmetic intensity | one-thread/output threads')
    for tile in tile_sizes:
        bytes_loaded = 2 * tile * tile * 4
        flops = 2 * tile**3
        intensity = flops / bytes_loaded
        print(f'{tile:4d} | {bytes_loaded / 1024:10.1f} KiB | {flops:8,d} | {intensity:7.1f} FLOPs/byte | {tile * tile:7,d}')

print_tile_model()

## 3. Benchmark helper

The reported value is achieved TFLOP/s, not the T4's theoretical peak:

$$\text{TFLOP/s}=\frac{2MNK}{T_{seconds}\times10^{12}}$$

In [ ]:
def benchmark_matmul(m, k, n, dtype=torch.float16, warmup=10, repeats=40):
    a = torch.randn((m, k), device=device, dtype=dtype)
    b = torch.randn((k, n), device=device, dtype=dtype)
    c = torch.empty((m, n), device=device, dtype=dtype)

    for _ in range(warmup):
        torch.mm(a, b, out=c)
    torch.cuda.synchronize(device)

    samples_ms = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        torch.mm(a, b, out=c)
        end.record()
        end.synchronize()
        samples_ms.append(start.elapsed_time(end))

    median_ms = statistics.median(samples_ms)
    flops = 2 * m * k * n
    tflops = flops / (median_ms / 1000) / 1e12
    return median_ms, tflops

def print_result(label, shape, dtype, result):
    ms, tflops = result
    print(f'{label:18s} shape={shape!s:20s} dtype={str(dtype).removeprefix("torch."):7s} time={ms:9.3f} ms TFLOP/s={tflops:7.2f}')

## 4. Experiment A — square FP16 GEMM

Small shapes often underutilize the GPU because launch overhead and incomplete tiles matter more. Larger shapes usually expose more parallel work, but throughput need not increase monotonically.

In [ ]:
sizes = [256, 512, 1024, 2048, 4096]
square_results = []
for n in sizes:
    result = benchmark_matmul(n, n, n, dtype=torch.float16)
    square_results.append((n, *result))
    print_result(f'N={n}', (n, n, n), torch.float16, result)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot([row[0] for row in square_results], [row[2] for row in square_results], marker='o')
plt.xlabel('Square matrix dimension')
plt.ylabel('Achieved TFLOP/s')
plt.title('NVIDIA T4 — FP16 GEMM throughput')
plt.grid(alpha=0.3)
plt.show()

## 5. Experiment B — FP16 versus FP32

FP16 can select T4 Tensor Core kernels when shape alignment and library heuristics permit it. This benchmark measures performance, but does not by itself prove which instructions executed.

In [ ]:
for dtype in (torch.float16, torch.float32):
    print(f'\nTesting {dtype}')
    for n in [512, 1024, 2048, 4096]:
        result = benchmark_matmul(n, n, n, dtype=dtype)
        print_result(f'N={n}', (n, n, n), dtype, result)

In [ ]:
torch.manual_seed(42)
a32 = torch.randn(512, 512, device=device, dtype=torch.float32)
b32 = torch.randn(512, 512, device=device, dtype=torch.float32)
c32 = a32 @ b32
c16 = a32.half() @ b32.half()
difference = (c32 - c16.float()).abs()
print('Mean absolute difference:', difference.mean().item())
print('Maximum absolute difference:', difference.max().item())

## 6. Experiment C — decode-like versus prefill-like shapes

Use a fixed $4096\times4096$ FP16 weight matrix shape and vary the token-row dimension $M$. The weight matrix remains 32 MiB, but larger $M$ values provide more opportunities for weight reuse and GPU utilization.

Predict before running: achieved TFLOP/s can increase as token count grows because the operation becomes more GEMM-like.

In [ ]:
token_counts = [1, 4, 16, 64, 256, 1024]
projection_results = []
for tokens in token_counts:
    result = benchmark_matmul(tokens, 4096, 4096, dtype=torch.float16)
    projection_results.append((tokens, *result))
    print_result(f'tokens={tokens}', (tokens, 4096, 4096), torch.float16, result)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot([row[0] for row in projection_results], [row[2] for row in projection_results], marker='o')
plt.xscale('log', base=2)
plt.xlabel('Token rows (M)')
plt.ylabel('Achieved TFLOP/s')
plt.title('Projection throughput versus token rows')
plt.grid(alpha=0.3)
plt.show()

## Interpretation and assignment

Answer these questions after recording your measurements:

1. Which square shapes achieved the highest TFLOP/s, and why might the smallest shapes be less efficient?
2. How did FP16 and FP32 latency differ on your T4?
3. Did projection TFLOP/s rise with token count? Explain using **arithmetic intensity**, **weight reuse**, and **GPU utilization**.
4. Where do memory movement, boundary tiles, or underutilization appear to matter most?
5. Why can higher aggregate throughput still coexist with higher latency for one individual request?

## Optional: naive CPU GEMM

This intentionally slow triple loop demonstrates the arithmetic structure. It is not a fair CPU-versus-GPU benchmark because Python interpreter overhead dominates. Keep the matrix small.

In [ ]:
# Optional: run only when you want to see the three nested loops.
import time
import numpy as np

n = 64
rng = np.random.default_rng(42)
a = rng.standard_normal((n, n), dtype=np.float32)
b = rng.standard_normal((n, n), dtype=np.float32)
c = np.zeros((n, n), dtype=np.float32)
started = time.perf_counter()
for i in range(n):
    for j in range(n):
        for k in range(n):
            c[i, j] += a[i, k] * b[k, j]
elapsed_ms = (time.perf_counter() - started) * 1000
print('Python loop time:', round(elapsed_ms, 2), 'ms')
print('Correct against NumPy:', np.allclose(c, a @ b, atol=1e-3))

### Final thought

The same weight matrix can produce very different performance depending on how many token rows reuse it. One decoding token is often GEMV-like and relatively memory-bound; 64 rows provide more arithmetic intensity, more weight reuse, and more work for the GPU to schedule. This is one reason continuous batching is important in vLLM.